# STATS 414 Midterm: Kalshi & Polymarket EDA

## Polymarket

### Library Imports Environment Variables, and Function Definitions

In [ ]:
## Library Imports
import numpy as np
import pandas as pd

from dotenv import load_dotenv
import os
import base64

import duckdb

from datetime import datetime, timedelta, date
import time

In [ ]:
## Load environment variables from the .env file
load_dotenv(override = True)  

POLYMARKET_URL = os.getenv('POLYMARKET_URL')
POLYMARKET_URL_BASE = os.getenv('POLYMARKET_URL_BASE')
POLYMARKET_USER = os.getenv('POLYMARKET_USER')
POLYMARKET_USER_SUFFIX = os.getenv('POLYMARKET_USER_SUFFIX')
POLYMARKET_PW = os.getenv('POLYMARKET_PW')
POLYMARKET_DB_HOST = os.getenv('POLYMARKET_DB_HOST')
POLYMARKET_DB_PORT = os.getenv('POLYMARKET_DB_PORT')

In [ ]:
SAMPLE_DATE = "2026-10-20"           # day of the order-book snapshot used for the liquidity chart
SAMPLE_HOUR_OF_DAY = 14              # 14:00 UTC = 07:00 in Los Angeles (PDT)

In [ ]:
# Basic-auth token for the HTTP secret below (the password is never printed or stored in a URL)
polymarketCloudFrontToken = base64.b64encode(f"{POLYMARKET_USER}:{POLYMARKET_PW}".encode()).decode()

In [ ]:
def trades_url(date: str) -> str:
    """
    
    Returns the URL for the trades file on the given date.
    Parameters:
        date (str): The trade date in 'YYYY-MM-DD' format.
    
    Returns:
        The URL for the trades file on the given date.
    """

    return f"https://{POLYMARKET_URL_BASE}/wallet_trades/trades-{date}.parquet"


def book_url(kind: str, hour: str) -> str:
    """
    Returns the URL for the book file of the given kind and hour.

    Parameters:
        kind (str): The type of book file, either 'events' (deltas) or 'snap' (snapshots).
        hour (str): The hour of the book file in 'YYYY-MM-DDTHH' format.

    Returns:
        The URL for the book file of the given kind and hour.
    """
    
    return f"https://{POLYMARKET_URL_BASE}/{kind}-{hour}.parquet"

In [ ]:
def hour_str(dt: datetime) -> str:
    """
    Returns the hour string for the given datetime in 'YYYY-MM-DDTHH' format.

    Parameters:
        dt (datetime): The datetime object.

    Returns:
        The hour string in 'YYYY-MM-DDTHH' format.
    """
    
    return dt.strftime("%Y-%m-%dT%H")

In [ ]:
def fetch_row(sql: str) -> list:
    """
    Returns the first row of a query as a Python list (DuckDB 2.0 relations have no fetchone()).

    Parameters:
        sql (str): The SQL query to execute.

    Returns:
        The first row of the query result as a Python list.
    """
    
    return duckdb.sql(sql).df().iloc[0].tolist()

In [ ]:
def try_count(url: str):
    """
    Returns the row count from parquet metadata (cheap); returns an error string if unreadable.

    Parameters:
        url (str): The URL of the parquet file.

    Returns:
        The row count if successful.
        An error string if the file is unreadable.
    """

    try:
        return int(fetch_row(f"SELECT count(*) FROM read_parquet('{url}')")[0])
    except Exception as e:  # noqa: BLE001
        return f"ERR: {str(e)[:100]}"

In [ ]:
plots: dict[str, pd.DataFrame] = {}

In [ ]:
def keep(name: str, sql: str, chart: str = "", n: int = 12):
    """
    Run a query, store the result in `plots[name]`, and print a preview.

    Parameters:
        name (str): The name to store the result under in `plots`.
        sql (str): The SQL query to execute.
        chart (str, optional): The chart type for the preview. Defaults to "".
        n (int, optional): The number of rows to preview. Defaults to 12.

    Returns:
        The result of the query as a DataFrame.
    """
    
    df = duckdb.sql(sql).df()
    plots[name] = df
    print(f"[{name}]  chart: {chart}  rows: {len(df)}")
    print(df.head(n))
    print()
    return df

In [ ]:
def date_range(start: str, end: str) -> list[str]:
    """
    Returns a list of dates between the start and end dates (inclusive) in 'YYYY-MM-DD' format.

    Parameters:
        start (str): The start date in 'YYYY-MM-DD' format.
        end (str): The end date in 'YYYY-MM-DD' format.

    Returns:
        A list of date strings in 'YYYY-MM-DD' format.
    """
    
    s, e = date.fromisoformat(start), date.fromisoformat(end)
    return [(s + timedelta(days=i)).isoformat() for i in range((e - s).days + 1)]

### Connections

In [ ]:
# HTTP access to the parquet files (order books and wallet trades)
duckdb.sql("INSTALL httpfs; LOAD httpfs;")

duckdb.sql(f"""
    CREATE OR REPLACE SECRET polymarket_http (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Basic {polymarketCloudFrontToken}'}}
    )
""")

In [ ]:
# Catalog database (markets, events, resolutions), attached as `pm`
duckdb.sql("INSTALL postgres; LOAD postgres;")

duckdb.sql(f"""
    ATTACH 'host={POLYMARKET_DB_HOST} port={POLYMARKET_DB_PORT} dbname=postgres
             user={POLYMARKET_USER}.{POLYMARKET_USER_SUFFIX} password={POLYMARKET_PW} sslmode=require'
    AS pm (TYPE postgres, READ_ONLY);
""")

### Local Copies of the Catalog
Small local tables that the later sections join against (timestamps normalised to naive UTC).

In [ ]:
# markets and settled markets
duckdb.sql("""
CREATE OR REPLACE TABLE mk AS
SELECT slug, title, event_slug, neg_risk, volume_clob,
       start_date AT TIME ZONE 'UTC' AS start_utc,
       end_date   AT TIME ZONE 'UTC' AS end_utc
FROM pm.markets
""")

duckdb.sql("""
CREATE OR REPLACE TABLE res AS
SELECT market_slug, resolved_yes, price_at_close,
       resolved_at AT TIME ZONE 'UTC' AS resolved_utc
FROM pm.resolutions
""")

# asset_id <-> market slug <-> YES/NO (token_ids is ordered YES then NO)
duckdb.sql("""
CREATE OR REPLACE TABLE token_map AS
SELECT asset_id, slug, condition_id, is_yes
FROM (
    SELECT slug, condition_id, json_extract_string(token_ids, '$[0]') AS asset_id, TRUE  AS is_yes
    FROM pm.markets WHERE json_array_length(token_ids) = 2
    UNION ALL
    SELECT slug, condition_id, json_extract_string(token_ids, '$[1]') AS asset_id, FALSE AS is_yes
    FROM pm.markets WHERE json_array_length(token_ids) = 2
)
""")

### Catalog Tables
Plots: `catalog__markets_opened_per_month`, `catalog__duration_hist`, `catalog__volume_pareto`, `catalog__yes_rate_by_neg_risk`

In [ ]:
keep("markets_opened_per_month", """
SELECT date_trunc('month', start_date) AS month, count(*) AS markets_opened
FROM pm.markets WHERE start_date IS NOT NULL
GROUP BY 1 ORDER BY 1
""", "line/bar: platform growth over time")

keep("duration_hist", """
SELECT floor(log10(epoch(end_date - start_date) / 86400.0) * 4) / 4 AS log10_days_bin,
       count(*) AS markets
FROM pm.markets WHERE end_date > start_date
GROUP BY 1 ORDER BY 1
""", "histogram, log x-axis: how long markets are scheduled to run (bin = 10^x days)")

keep("volume_pareto", """
WITH r AS (
    SELECT volume_clob,
           row_number() OVER (ORDER BY volume_clob DESC, slug)                          AS rank,
           sum(volume_clob) OVER (ORDER BY volume_clob DESC, slug ROWS UNBOUNDED PRECEDING) AS cum_volume,
           sum(volume_clob) OVER ()                                                      AS total_volume,
           count(*) OVER ()                                                              AS n
    FROM pm.markets WHERE volume_clob > 0
)
SELECT rank, rank * 1.0 / n AS market_percentile, cum_volume / total_volume AS cum_volume_share
FROM r WHERE rank <= 2000 OR rank % 50 = 0
ORDER BY rank
""", "Pareto curve: cumulative share of volume by market rank", n=5)

keep("yes_rate_by_neg_risk", """
SELECT m.neg_risk, count(*) AS resolved_markets, avg(r.resolved_yes::INT) AS yes_rate
FROM pm.resolutions r JOIN pm.markets m ON m.slug = r.market_slug
GROUP BY 1 ORDER BY 1
""", "bars: share of markets where YES won")

### Order Book: Liquidity at One Snapshot
Plot: `orderbook__liquidity_by_volume` (the order book at 14:00 UTC on the sample day)

In [ ]:
sample_dt = datetime.strptime(SAMPLE_DATE, "%Y-%m-%d").replace(hour=SAMPLE_HOUR_OF_DAY)
HOUR = hour_str(sample_dt)

duckdb.sql(f"CREATE OR REPLACE TABLE snap_h AS SELECT * FROM read_parquet('{book_url('snap', HOUR)}')")

In [ ]:
duckdb.sql("""
CREATE OR REPLACE TABLE book_stats AS
SELECT s.asset_id, s.market, t.slug, t.is_yes,
       len(bids) AS n_bids, len(asks) AS n_asks,
       bids[1].price AS best_bid, asks[1].price AS best_ask,
       asks[1].price - bids[1].price       AS spread,
       (asks[1].price + bids[1].price) / 2 AS mid,
       list_sum(list_transform(bids, lambda x: x.size)) AS bid_size_total,
       list_sum(list_transform(asks, lambda x: x.size)) AS ask_size_total,
       list_sum(list_transform(list_filter(bids, lambda x: x.price >= bids[1].price - 0.05), lambda x: x.size)) AS bid_depth_5c,
       list_sum(list_transform(list_filter(asks, lambda x: x.price <= asks[1].price + 0.05), lambda x: x.size)) AS ask_depth_5c
FROM snap_h s LEFT JOIN token_map t USING (asset_id)
""")

In [ ]:
# Cost of trading vs market size, in cents per share, by actual trading volume (log-spaced groups: $1K-3K, $3K-10K, ...).
# "Extra cost" = walk up the YES sell-side book and see how much more than the best sell price you would pay on
# average to buy $100 (or $1,000) worth of YES shares right now. NULL when the book cannot fill that much.
MIN_BOOKS_PER_VOLUME_GROUP = 20
keep("liquidity_by_volume", f"""
WITH yes_books AS (
    SELECT bs.asset_id, bs.spread, bs.best_ask, m.volume_clob
    FROM book_stats bs JOIN pm.markets m ON m.slug = bs.slug
    WHERE bs.is_yes AND bs.n_bids > 0 AND bs.n_asks > 0 AND bs.spread >= 0
      AND m.volume_clob >= 1 AND bs.best_ask > 0
),
ask_levels AS (
    SELECT asset_id, u.price AS price, u.size AS size
    FROM (SELECT asset_id, unnest(asks) AS u FROM snap_h WHERE asset_id IN (SELECT asset_id FROM yes_books))
    WHERE u.price > 0
),
walk AS (
    SELECT asset_id, price, size,
           coalesce(sum(price * size) OVER (PARTITION BY asset_id ORDER BY price
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS dollars_before
    FROM ask_levels
),
fill AS (
    SELECT asset_id,
           sum(least(price * size, greatest(100  - dollars_before, 0)))         AS spent_100,
           sum(least(price * size, greatest(100  - dollars_before, 0)) / price) AS shares_100,
           sum(least(price * size, greatest(1000 - dollars_before, 0)))         AS spent_1000,
           sum(least(price * size, greatest(1000 - dollars_before, 0)) / price) AS shares_1000
    FROM walk GROUP BY asset_id
),
per_book AS (
    SELECT y.volume_clob, y.spread * 100 AS gap_cents,
           CASE WHEN f.spent_100  >= 100  - 1e-6 THEN (f.spent_100  / f.shares_100  - y.best_ask) * 100 END AS extra_100,
           CASE WHEN f.spent_1000 >= 1000 - 1e-6 THEN (f.spent_1000 / f.shares_1000 - y.best_ask) * 100 END AS extra_1000,
           floor(log10(y.volume_clob) * 2) / 2 AS log10_bin
    FROM yes_books y JOIN fill f USING (asset_id)
)
SELECT power(10, log10_bin) AS volume_lo, power(10, log10_bin + 0.5) AS volume_hi,
       count(*) AS books,
       median(gap_cents) AS median_gap_cents,
       avg((extra_100  IS NOT NULL)::INT) AS share_can_fill_100,
       avg((extra_1000 IS NOT NULL)::INT) AS share_can_fill_1000,
       CASE WHEN avg((extra_100  IS NOT NULL)::INT) >= 0.5 THEN median(extra_100)  END AS median_extra_100_cents,
       CASE WHEN avg((extra_1000 IS NOT NULL)::INT) >= 0.5 THEN median(extra_1000) END AS median_extra_1000_cents
FROM per_book GROUP BY log10_bin
HAVING count(*) >= {MIN_BOOKS_PER_VOLUME_GROUP}
ORDER BY log10_bin
""", "two stacked panels by actual volume (log groups): (1) bid-ask gap in cents, (2) extra cents per share to buy "
     "$100 and $1,000 of YES; label each group with its number of books")

### Wallet Trades
Plots: `wallets__daily_overview`, `wallets__dow_hour_heatmap`, `wallets__wallet_lorenz`, `wallets__burner_buckets`,
`wallets__top_markets`, `wallets__topic_share_by_day`, `wallets__time_to_close`

In [ ]:
DAILY_START = "2026-10-01"   # range for the cheap day-by-day overview (extend backwards if you like)
DAILY_END = "2026-10-26"     # today's file (10-27) is still partial
SAMPLE_START = "2026-10-14"  # window loaded locally for the deeper analysis (inclusive)
SAMPLE_END = "2026-10-20"
MAX_ROWS_TO_LOAD = 50_000_000

In [ ]:
# `time` may be TIMESTAMP or TIMESTAMPTZ in the parquet - normalise to naive UTC either way.
probe_url = trades_url(SAMPLE_END)
time_type = duckdb.sql(f"DESCRIBE SELECT \"time\" FROM read_parquet('{probe_url}')").df()["column_type"][0]
TIME_EXPR = "\"time\" AT TIME ZONE 'UTC'" if "WITH TIME ZONE" in time_type else "\"time\""
print("time column type:", time_type, "->", TIME_EXPR)

In [ ]:
# One cheap query per daily file (remote aggregation, nothing is loaded locally)
rows = []
for d in date_range(DAILY_START, DAILY_END):
    try:
        r = fetch_row(
            f"SELECT count(*), sum(dollars), count(DISTINCT wallet), count(DISTINCT market) "
            f"FROM read_parquet('{trades_url(d)}')"
        )
        rows.append({"date": d, "trades": r[0], "dollars": r[1], "active_wallets": r[2], "active_markets": r[3]})
    except Exception as e:  # noqa: BLE001
        rows.append({"date": d, "trades": None, "dollars": None, "active_wallets": None,
                     "active_markets": None, "error": str(e)[:80]})
daily = pd.DataFrame(rows)
daily["avg_trade_usd"] = daily["dollars"] / daily["trades"]
plots["daily_overview"] = daily  # chart: line / area, dollars and active wallets over time (dual axis)
print("[daily_overview]  chart: line, dollars + active wallets per day  rows:", len(daily))

In [ ]:
# Trades for the analysis window (loaded locally)
sample_days = date_range(SAMPLE_START, SAMPLE_END)
urls = [trades_url(d) for d in sample_days]
counts = [try_count(u) for u in urls]
print(dict(zip(sample_days, counts)))
assert all(isinstance(c, int) for c in counts), "Some files in the window are unreadable - see counts above"
assert sum(counts) <= MAX_ROWS_TO_LOAD, f"Too many rows to load locally: {sum(counts):,}"

url_list = ", ".join(f"'{u}'" for u in urls)
duckdb.sql(f"""
CREATE OR REPLACE TABLE trades AS
SELECT wallet, market, {TIME_EXPR} AS ts, side, outcome, price, size, dollars,
       regexp_extract(filename, 'trades-(\\d{{4}}-\\d{{2}}-\\d{{2}})', 1) AS file_date
FROM read_parquet([{url_list}], filename = true)
""")

In [ ]:
keep("dow_hour_heatmap", """
SELECT dayofweek(ts) AS day_of_week_sun0, hour(ts) AS hour_utc, sum(dollars) AS dollars, count(*) AS trades
FROM trades GROUP BY 1, 2 ORDER BY 1, 2
""", "heatmap: day of week x hour of day")

In [ ]:
# Per-wallet totals -> Lorenz curve and burner buckets
duckdb.sql("""
CREATE OR REPLACE TABLE wallet_stats AS
SELECT wallet, count(*) AS trades, count(DISTINCT market) AS markets, sum(dollars) AS dollars,
       avg(dollars) AS avg_trade_usd, avg((side = 'BUY')::INT) AS buy_share,
       min(ts) AS first_ts, max(ts) AS last_ts
FROM trades GROUP BY wallet
""")

keep("wallet_lorenz", """
WITH r AS (SELECT dollars, ntile(100) OVER (ORDER BY dollars) AS pct FROM wallet_stats),
     b AS (SELECT pct, sum(dollars) AS dollars FROM r GROUP BY 1)
SELECT pct AS wallet_percentile,
       sum(dollars) OVER (ORDER BY pct) / sum(dollars) OVER () AS cum_dollar_share
FROM b ORDER BY pct
""", "Lorenz curve vs the 45-degree line: how concentrated is trading across wallets?", n=5)

keep("burner_buckets", """
SELECT CASE WHEN markets = 1 THEN '1_one_market'
            WHEN markets <= 5 THEN '2_2-5'
            WHEN markets <= 20 THEN '3_6-20'
            WHEN markets <= 100 THEN '4_21-100'
            ELSE '5_100+' END AS markets_traded,
       count(*) AS wallets, sum(dollars) AS dollars, median(dollars) AS median_dollars_per_wallet
FROM wallet_stats GROUP BY 1 ORDER BY 1
""", "bars: wallets vs dollars by breadth (one-market 'burners' vs diversified)")

In [ ]:
# Per-market totals -> the 20 most traded markets
duckdb.sql("""
CREATE OR REPLACE TABLE market_stats AS
SELECT t.market, any_value(m.title) AS title, count(*) AS trades, count(DISTINCT t.wallet) AS wallets,
       sum(t.dollars) AS dollars
FROM trades t LEFT JOIN mk m ON m.slug = t.market
GROUP BY t.market
""")

keep("top_markets", """
SELECT title, trades, wallets, round(dollars) AS dollars FROM market_stats ORDER BY dollars DESC LIMIT 20
""", "horizontal bar of the 20 most traded markets")

In [ ]:
# Rough topic buckets from titles (keyword heuristic - refine after eyeballing 'other')
keep("topic_share_by_day", r"""
WITH t AS (
    SELECT strftime(tr.ts, '%Y-%m-%d') AS day, tr.dollars,
           CASE
             WHEN regexp_matches(lower(m.title), '\b(bitcoin|btc|ethereum|eth|solana|sol|xrp|crypto|dogecoin)\b') THEN 'crypto'
             WHEN regexp_matches(lower(m.title), '\b(nfl|nba|mlb|nhl|ufc|fifa|premier league|champions league|world cup|tennis|golf|f1|vs\.?)\b') THEN 'sports'
             WHEN regexp_matches(lower(m.title), '\b(trump|biden|harris|election|president|senate|congress|democrat|republican|governor|primary|vote)\b') THEN 'politics'
             WHEN regexp_matches(lower(m.title), '\b(fed|interest rate|inflation|cpi|gdp|recession|tariff|unemployment)\b') THEN 'macro'
             ELSE 'other'
           END AS topic
    FROM trades tr LEFT JOIN mk m ON m.slug = tr.market
)
SELECT day, topic, sum(dollars) AS dollars, count(*) AS trades FROM t GROUP BY 1, 2 ORDER BY 1, 2
""", "stacked area / 100% bars: topic mix over time")

In [ ]:
keep("time_to_close", """
WITH j AS (
    SELECT t.dollars, epoch(r.resolved_utc - t.ts) / 3600.0 AS hours_to_close
    FROM trades t JOIN res r ON r.market_slug = t.market
)
SELECT CASE WHEN hours_to_close < 0    THEN '0_after_close'
            WHEN hours_to_close < 1    THEN '1_<1h'
            WHEN hours_to_close < 6    THEN '2_1-6h'
            WHEN hours_to_close < 24   THEN '3_6-24h'
            WHEN hours_to_close < 72   THEN '4_1-3d'
            WHEN hours_to_close < 168  THEN '5_3-7d'
            WHEN hours_to_close < 720  THEN '6_7-30d'
            ELSE '7_30d+' END AS time_before_close,
       count(*) AS trades, sum(dollars) AS dollars
FROM j GROUP BY 1 ORDER BY 1
""", "bars: how much trading happens in the final hours/days (and after close?)")

### Taker Edge (Held to Settlement)
Plots: `cross__edge_by_time_to_close`, `cross__edge_by_trade_size`, `cross__edge_by_wallet_class`.
taker_pnl = size * (won - price) for BUY, size * (price - won) for SELL; fees and early exits are ignored.

In [ ]:
# Time-before-settlement buckets (hours_to_close >= 0); also used by later cells
TTC_CASE = """CASE WHEN hours_to_close < 1   THEN '1_<1h'
                    WHEN hours_to_close < 6   THEN '2_1-6h'
                    WHEN hours_to_close < 24  THEN '3_6-24h'
                    WHEN hours_to_close < 72  THEN '4_1-3d'
                    WHEN hours_to_close < 168 THEN '5_3-7d'
                    WHEN hours_to_close < 720 THEN '6_7-30d'
                    ELSE '7_30d+' END"""

duckdb.sql(f"""
CREATE OR REPLACE TABLE trade_edge AS
WITH wc AS (SELECT wallet, count(DISTINCT market) AS markets FROM trades GROUP BY 1),
j AS (
    SELECT t.wallet, t.market, t.ts, t.side, t.size, t.price, t.dollars,
           ((t.outcome = 'Yes' AND r.resolved_yes) OR (t.outcome = 'No' AND NOT r.resolved_yes))::INT AS won,
           epoch(r.resolved_utc - t.ts) / 3600.0 AS hours_to_close
    FROM trades t JOIN res r ON r.market_slug = t.market
    WHERE t.outcome IN ('Yes', 'No') AND t.size > 0 AND t.dollars > 0
)
SELECT j.*, wc.markets AS wallet_markets,
       CASE WHEN j.side = 'BUY' THEN j.size * (j.won - j.price) ELSE j.size * (j.price - j.won) END AS taker_pnl,
       {TTC_CASE} AS time_before_close,
       CASE WHEN j.dollars < 10 THEN '1_<$10' WHEN j.dollars < 100 THEN '2_$10-100'
            WHEN j.dollars < 1000 THEN '3_$100-1k' WHEN j.dollars < 10000 THEN '4_$1k-10k'
            ELSE '5_$10k+' END AS size_bucket,
       CASE WHEN wc.markets = 1 THEN '1_one_market' WHEN wc.markets <= 5 THEN '2_2-5'
            WHEN wc.markets <= 20 THEN '3_6-20' ELSE '4_21+' END AS wallet_class
FROM j JOIN wc USING (wallet)
WHERE j.hours_to_close >= 0
""")

In [ ]:
keep("edge_by_time_to_close", """
SELECT time_before_close, side, count(*) AS trades, sum(dollars) AS dollars,
       sum(taker_pnl) / sum(dollars) AS return_per_dollar,
       avg(taker_pnl / size) AS avg_edge_per_share, stddev(taker_pnl / size) AS sd_edge_per_share
FROM trade_edge GROUP BY 1, 2 ORDER BY 1, 2
""", "grouped bars (+ error bars from sd/sqrt(trades)): are late traders better informed?")

keep("edge_by_trade_size", """
SELECT size_bucket, side, count(*) AS trades, sum(dollars) AS dollars,
       sum(taker_pnl) / sum(dollars) AS return_per_dollar, avg(taker_pnl / size) AS avg_edge_per_share
FROM trade_edge GROUP BY 1, 2 ORDER BY 1, 2
""", "grouped bars: do large trades beat small ones?")

keep("edge_by_wallet_class", """
SELECT wallet_class, count(*) AS trades, count(DISTINCT wallet) AS wallets, sum(dollars) AS dollars,
       sum(taker_pnl) / sum(dollars) AS return_per_dollar, avg(taker_pnl / size) AS avg_edge_per_share
FROM trade_edge GROUP BY 1 ORDER BY 1
""", "bars: one-market 'burners' vs diversified wallets")

### Order-Book Calibration
Plot: `cross__calibration_book`. For every market that settled in the window, take the YES price from the order-book snapshot
1, 6 and 24 hours before settlement and compare it with who actually won.

In [ ]:
RES_START = "2026-10-01 00:00:00"      # markets settled in [RES_START, RES_END) are used
RES_END = "2026-10-21 00:00:00"
HORIZONS_H = [1, 6, 24]                # book snapshot taken between h and h+1 hours before settlement
MAX_SPREAD = 0.2                       # ignore books whose spread is wider than this (mid not meaningful)
MAX_BOOK_FILES = 600                   # Oct 1-20 needs about 500 hourly snapshot files
CAL_TIME_BUDGET_MIN = 40               # stop loading after this many minutes (newest days first; rerun to continue)

In [ ]:
# Hours needed: the snapshots h..h+1 hours before each settlement in the window
need_parts = [f"""
    SELECT CAST(date_trunc('day', resolved_utc) AS DATE) AS day,
           strftime(date_trunc('hour', resolved_utc - INTERVAL {h} HOUR), '%Y-%m-%dT%H') AS hr
    FROM res
    WHERE resolved_utc >= TIMESTAMP '{RES_START}' AND resolved_utc < TIMESTAMP '{RES_END}'
    GROUP BY ALL""" for h in HORIZONS_H]
need_df = duckdb.sql(" UNION ".join(need_parts)).df()
order = sorted(set(need_df["hr"]), reverse=True)            # newest first
print(f"{len(order)} snapshot hours needed")
assert len(order) <= MAX_BOOK_FILES, f"Too many snapshot files ({len(order)}); narrow RES_START/RES_END or HORIZONS_H"

# Only the best level of each book is kept locally
duckdb.sql("""
CREATE TABLE IF NOT EXISTS book_top (
    hour VARCHAR, snap_ts TIMESTAMP, asset_id VARCHAR, market VARCHAR,
    best_bid DOUBLE, best_ask DOUBLE, best_bid_size DOUBLE, best_ask_size DOUBLE
)
""")
already = set(duckdb.sql("SELECT DISTINCT hour FROM book_top").df()["hour"])
t_load, n_loaded, out_of_time = time.time(), 0, False
for hr in order:
    if hr in already:
        continue
    if n_loaded >= 5 and (time.time() - t_load) / 60 > CAL_TIME_BUDGET_MIN:
        out_of_time = True
        break
    t0 = time.time()
    try:
        duckdb.sql(f"""
            INSERT INTO book_top
            SELECT '{hr}', epoch_ms(ts_ms), asset_id, market,
                   bids[1].price, asks[1].price, bids[1].size, asks[1].size
            FROM read_parquet('{book_url('snap', hr)}')
        """)
        n_loaded += 1
        print(f"loaded {hr} in {time.time() - t0:.1f}s")
        if n_loaded == 5:
            per = (time.time() - t_load) / 5
            left = len([x for x in order if x not in already]) - 5
            print(f"   ~{per:.1f}s per file -> about {per * left / 60:.0f} more minutes for the remaining {left} files")
    except Exception as e:  # noqa: BLE001
        print(f"FAILED {hr}: {str(e)[:120]}")
if out_of_time:
    print(f"Stopped loading after {CAL_TIME_BUDGET_MIN} min; the calibration uses only the newest days with enough snapshots. "
          "Run this cell again to continue where it stopped.")

# A settlement day is used when at least 90% of its needed snapshot hours are loaded
loaded_hours = set(duckdb.sql("SELECT DISTINCT hour FROM book_top").df()["hour"])
day_ok = need_df.groupby("day")["hr"].apply(lambda hs: len(set(hs) & loaded_hours) / len(set(hs)) >= 0.9)
ok_days = [pd.Timestamp(d).date() for d in sorted(day_ok[day_ok].index)]
all_days = [pd.Timestamp(d).date() for d in sorted(day_ok.index)]
print(f"Calibration days with snapshots: {len(ok_days)} of {len(all_days)}")
assert ok_days, "No settlement day has enough snapshot hours loaded - see the FAILED lines above"
CAL_WINDOW = {"window_start": ok_days[0].isoformat(), "window_end": ok_days[-1].isoformat(),
              "days_loaded": len(ok_days), "days_in_window": len(all_days)}
OK_DAY_SQL = ", ".join(f"DATE '{d}'" for d in ok_days)

In [ ]:
# YES price (book mid) at each horizon, next to who actually won
parts = []
for h in HORIZONS_H:
    parts.append(f"""
    SELECT r.market_slug AS slug, {h} AS horizon_h, r.resolved_yes,
           bt.best_bid, bt.best_ask, (bt.best_bid + bt.best_ask) / 2 AS mid,
           bt.best_ask - bt.best_bid AS spread
    FROM res r
    JOIN token_map tm ON tm.slug = r.market_slug AND tm.is_yes
    JOIN book_top bt ON bt.asset_id = tm.asset_id
                    AND bt.snap_ts = date_trunc('hour', r.resolved_utc - INTERVAL {h} HOUR)
    WHERE r.resolved_utc >= TIMESTAMP '{RES_START}' AND r.resolved_utc < TIMESTAMP '{RES_END}'
      AND CAST(date_trunc('day', r.resolved_utc) AS DATE) IN ({OK_DAY_SQL})
      AND bt.best_bid IS NOT NULL AND bt.best_ask IS NOT NULL AND bt.best_ask >= bt.best_bid
    """)
duckdb.sql("CREATE OR REPLACE TABLE cal_points AS " + " UNION ALL ".join(parts))


def stamp(name):
    """Add the settlement window actually used (days with enough snapshots) to a table."""
    plots[name] = plots[name].assign(**CAL_WINDOW)


# Ten price buckets (0-10%, 10-20%, ...). For each bucket, how often YES really won.
keep("calibration_book", f"""
SELECT horizon_h, round(least(floor(mid * 10), 9) / 10, 1) AS price_bucket_lo,
        count(*) AS markets, avg(mid) AS avg_price,
        sum(resolved_yes::INT) AS yes_won, avg(resolved_yes::INT) AS win_rate
FROM cal_points WHERE spread <= {MAX_SPREAD}
GROUP BY 1, 2 ORDER BY 1, 2
""", "one panel per horizon, 10 price buckets")
stamp("calibration_book")

## Export Parquet Files for the Plots

In [ ]:
# One parquet file per plot, named <prefix>__<table>.parquet - exactly what polymarket_plot_all_2.py reads.
os.makedirs("plot_data", exist_ok=True)
EXPORT = {
    "catalog":   ["volume_pareto", "yes_rate_by_neg_risk", "duration_hist", "markets_opened_per_month"],
    "wallets":   ["wallet_lorenz", "dow_hour_heatmap", "daily_overview", "time_to_close", "burner_buckets",
                  "top_markets", "topic_share_by_day"],
    "cross":     ["edge_by_time_to_close", "edge_by_trade_size", "edge_by_wallet_class", "calibration_book"],
    "orderbook": ["liquidity_by_volume"],
}
summary = []
for prefix, names in EXPORT.items():
    for name in names:
        if name not in plots:
            summary.append({"file": f"{prefix}__{name}.parquet", "rows": "MISSING - run its cell first"})
            continue
        duckdb.register("p", plots[name])
        duckdb.sql(f"COPY (SELECT * FROM p) TO 'plot_data/{prefix}__{name}.parquet' (FORMAT parquet)")
        duckdb.unregister("p")
        summary.append({"file": f"{prefix}__{name}.parquet", "rows": len(plots[name])})
print(pd.DataFrame(summary).to_string(index=False))